# Bengaluru & Chennai — Property & Urban Intelligence

Companion notebook to
[AI-PROPERTY-URBAN-INTELLIGENCE](https://github.com/ujwal-m-2006/AI-PROPERTY-URBAN-INTELLIGENCE).

This notebook does three things:

1. Loads all 17 platform CSVs plus both property datasets, from public URLs.
2. Reproduces the project's headline ML finding — that **most of the reported
   accuracy in a naive setup is geographic leakage**.
3. Shows the two results that are reported as failures rather than hidden.

**Nothing here is fitted to look good.** Two of the numbers you are about to see
are bad, and they are the interesting ones.

Runtime: a few minutes on a free CPU instance. No GPU, no Drive mount.

## 1 · Setup

In [ ]:
import warnings, io, json, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2 · Load the platform layers

All 17 CSVs come from the repository. Each file's first line is a `#` comment
carrying the caveat that constrains how the table may be read — so the loader
keeps it rather than skipping past it.

If you would rather use your own Drive folder, set `BASE` to that path instead
of the URL.

In [ ]:
BASE = "https://raw.githubusercontent.com/ujwal-m-2006/AI-PROPERTY-URBAN-INTELLIGENCE/main/colab/data"          # or e.g. "/content/drive/MyDrive/colab_data"

LAYERS = [
    "gba_wards", "gba_corporations", "chennai_wards", "chennai_zones",
    "admin_taluks_bengaluru", "admin_taluks_chennai",
    "revenue_parcels", "road_network_bengaluru", "flood_locations_bengaluru",
    "localities_bengaluru", "localities_chennai",
    "amenities_bengaluru", "amenities_chennai",
    "ward_analytics_bengaluru", "ward_analytics_chennai",
    "model_comparison", "_sources",
]

def load(name):
    """Read a layer, keeping its caveat line."""
    path = f"{BASE}/{name}.csv"
    caveat = ""
    try:
        head = pd.read_csv(path, nrows=0, header=None, comment=None).columns
    except Exception:
        pass
    df = pd.read_csv(path, comment="#", low_memory=False)
    # Recover the caveat comment, which pandas skipped.
    try:
        import urllib.request
        first = urllib.request.urlopen(path).readline().decode("utf-8", "ignore")
        if first.startswith("#"):
            caveat = first.lstrip("# ").strip()
    except Exception:
        try:
            with open(path, encoding="utf-8") as fh:
                first = fh.readline()
            if first.startswith("#"):
                caveat = first.lstrip("# ").strip()
        except Exception:
            caveat = ""
    df.attrs["caveat"] = caveat
    return df

data = {}
for name in LAYERS:
    try:
        data[name] = load(name)
        print(f"  {name:<32} {len(data[name]):>6,} rows x {data[name].shape[1]:>2} cols")
    except Exception as exc:
        print(f"  {name:<32} FAILED: {type(exc).__name__}")

print(f"\n{sum(len(d) for d in data.values()):,} rows loaded across {len(data)} layers")

### The caveats are the point

Several of these tables would mislead if read at face value. They say so
themselves.

In [ ]:
for name in ["gba_wards", "road_network_bengaluru", "revenue_parcels",
             "flood_locations_bengaluru", "chennai_wards"]:
    if name in data and data[name].attrs.get("caveat"):
        print(f"── {name}")
        print(f"   {data[name].attrs['caveat'][:300]}\n")

## 3 · Provenance — where every layer came from

In [ ]:
src = data["_sources"][["layer", "organisation", "tier", "availability",
                        "licence", "verification_status"]]
display(src)

print("\nTier meaning: T1 official primary · T2 official republished · "
      "T3 community/open · T4 commercial dataset")
print("No layer here is T1 — every government layer reached this project "
      "through a portal or a community mirror, and is capped accordingly.")

## 4 · The two-width problem

`road_network_bengaluru` carries **two** width columns. Feeding the wrong one
into a floor-area calculation would roughly double the buildable area for the
whole city, with nothing on screen looking wrong.

In [ ]:
rd = data["road_network_bengaluru"]
w = rd[["width_existing_m", "width_proposed_m"]].describe().loc[["min", "50%", "max"]]
display(w)

larger = (rd["width_proposed_m"] > rd["width_existing_m"]).mean()
print(f"proposed > existing on {larger:.1%} of {len(rd):,} segments")
print("A file titled 'road width map' contains a road-widening PROPOSAL.")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(rd["width_existing_m"].dropna(), bins=40, alpha=.75, label="existing")
ax.hist(rd["width_proposed_m"].dropna(), bins=40, alpha=.55, label="proposed")
ax.set_xlabel("road width (m)"); ax.set_ylabel("segments"); ax.legend()
ax.set_title("Two width columns, one filename")
plt.tight_layout(); plt.show()

## 5 · The suppressed population column

`gba_wards._raw_tot_p` is the source's own population field. It sums to roughly
six times the real Greater Bengaluru total, so the platform publishes **no**
ward population at all.

Dividing by 6 reproduces the officially reported per-corporation averages almost
exactly — including East's distinctly lower figure, which nothing was fitted to.
That is suggestive, and it is still not enough.

In [ ]:
gw = data["gba_wards"]
print(f"sum of _raw_tot_p : {gw['_raw_tot_p'].sum():,}")
print(f"real Greater Bengaluru population is roughly 14,000,000")
print(f"sum / 6           : {gw['_raw_tot_p'].sum()/6:,.0f}\n")

by_corp = gw.groupby("corporation")["_raw_tot_p"].agg(["count", "mean"])
by_corp["mean_div_6"] = (by_corp["mean"] / 6).round(0)
by_corp["reported"] = by_corp.index.map(
    {"Central": 40000, "East": 26000, "North": 40000,
     "South": 40000, "West": 40000})
display(by_corp)

inconsistent = (gw["_raw_tot_m"] + gw["_raw_tot_f"] != gw["_raw_tot_p"]).sum()
print(f"\nrows where male + female != total: {inconsistent} of {len(gw)}")
print("A constant chosen because it makes a total look right is not provenance,")
print("and a third of the rows fail an internal check unrelated to scale.")
print("=> the platform publishes no ward population.")

## 6 · The property datasets

Fetched from their original public sources, not from this repository — the
platform never redistributes them.

* Bengaluru — **asking** prices from listings
* Chennai — **recorded sale** prices

Those targets measure different things and are never pooled.

In [ ]:
blr_raw = pd.read_csv("https://raw.githubusercontent.com/dphi-official/Datasets/master/Bengaluru_House_Data.csv")
chn_raw = pd.read_csv("https://raw.githubusercontent.com/Ravi8149/Chennai-House-Price-Prediction/HEAD/chennai-house-price.csv")
print("Bengaluru", blr_raw.shape, "| Chennai", chn_raw.shape)
display(blr_raw.head(3))
display(chn_raw.head(3))

### Cleaning

Only what is needed to reproduce the headline result: parse the messy area
column, derive price per sq.ft, drop the extreme tail.

In [ ]:
def parse_sqft(v):
    """total_sqft holds ranges ('1133 - 1384') and units ('34.46Sq. Meter')."""
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if "-" in s:
        parts = s.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return np.nan
    try:
        return float(s)
    except ValueError:
        pass
    import re
    m = re.match(r"([\d.]+)\s*(\D+)", s)
    if not m:
        return np.nan
    n, unit = float(m.group(1)), m.group(2).lower()
    factor = {"sq. meter": 10.7639, "sq. yards": 9.0, "perch": 272.25,
              "acres": 43560.0, "cents": 435.6, "guntha": 1089.0,
              "grounds": 2400.0}
    for k, f in factor.items():
        if k in unit:
            return n * f
    return np.nan

blr = blr_raw.copy()
blr["sqft"] = blr["total_sqft"].apply(parse_sqft)
blr["rooms"] = blr["size"].astype(str).str.extract(r"(\d+)").astype(float)
blr["price_inr"] = blr["price"] * 1e5                 # source is in lakh
blr["price_per_sqft"] = blr["price_inr"] / blr["sqft"]
blr["locality"] = blr["location"].astype(str).str.strip()
blr = blr.dropna(subset=["price_per_sqft", "sqft", "rooms", "locality"])
lo, hi = blr["price_per_sqft"].quantile([0.01, 0.99])
blr = blr[(blr["price_per_sqft"] >= lo) & (blr["price_per_sqft"] <= hi)]
blr = blr[(blr["sqft"] > 100) & (blr["sqft"] < 30000)]

print(f"Bengaluru after cleaning: {len(blr):,} rows, "
      f"{blr['locality'].nunique()} localities")
print(f"  median asking: Rs {blr['price_per_sqft'].median():,.0f}/sq.ft")

## 7 · The headline finding — random k-fold leaks

A property and its next-door neighbour are nearly the same row. Random k-fold
puts them on opposite sides of the split, so the model memorises each locality's
price level and reports it as skill.

Grouping the folds by locality removes that. The gap is the leakage.

In [ ]:
FEATURES_NUM = ["sqft", "rooms", "bath", "balcony"]
FEATURES_CAT = ["area_type", "availability"]

X = blr[FEATURES_NUM + FEATURES_CAT]
y = blr["price_per_sqft"].to_numpy()
groups = blr["locality"].to_numpy()

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), FEATURES_NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore",
                                           min_frequency=10))]), FEATURES_CAT),
])

MODELS = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=120, random_state=42,
                                           n_jobs=-1),
    "gradient_boosting": GradientBoostingRegressor(random_state=42),
}

rows = []
for name, est in MODELS.items():
    pipe = Pipeline([("pre", pre), ("m", est)])
    rand = cross_val_score(pipe, X, y, cv=KFold(5, shuffle=True,
                                                random_state=42),
                           scoring="r2", n_jobs=-1).mean()
    spat = cross_val_score(pipe, X, y, cv=GroupKFold(5), groups=groups,
                           scoring="r2", n_jobs=-1).mean()
    rows.append({"algorithm": name, "random_cv_r2": round(rand, 4),
                 "spatial_cv_r2": round(spat, 4),
                 "leakage_gap": round(rand - spat, 4)})
    print(f"  {name:<20} random {rand:.4f}   spatial {spat:.4f}   "
          f"gap {rand-spat:+.4f}")

leak = pd.DataFrame(rows)
display(leak)
print("\nEvery model scores higher under random CV. The gap is the part of the")
print("accuracy that was leakage. Model selection must use the lower number.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
i = np.arange(len(leak))
ax.bar(i - 0.2, leak["random_cv_r2"], 0.4, label="random k-fold (optimistic)")
ax.bar(i + 0.2, leak["spatial_cv_r2"], 0.4, label="grouped by locality (honest)")
ax.set_xticks(i); ax.set_xticklabels(leak["algorithm"], rotation=15, ha="right")
ax.set_ylabel("R²"); ax.legend(); ax.set_title("The same models, two validation schemes")
ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

## 8 · Chennai — and a trap inside the trap

Chennai's dataset covers only **7 localities**, so there are only 7 spatial
blocks. Hold out a whole locality and the models fall apart, while a random
split still reports respectable accuracy.

But there is a subtler problem first, and an earlier version of this notebook
walked straight into it. The `AREA` column is **misspelt**: `Chrompet` also
appears as `Chrompt`, `Chrmpet` and `Chormpet`; `Anna Nagar` as `Ana Nagar` and
`Ann Nagar`. Group by the raw column and you get **17** groups instead of 7 — so
the same locality lands on *both* sides of the split, and the leakage this
notebook exists to measure quietly leaks back in.

The mapping below is the one the platform's `ml/pipelines/city_config.py`
applies, not an ad-hoc guess.

In [ ]:
chn = chn_raw.copy()
chn.columns = [c.strip().upper() for c in chn.columns]

# Typo'd categoricals, normalised exactly as the platform pipeline does.
AREA_FIXES = {
    "Karapakam": "Karapakkam", "Ana Nagar": "Anna Nagar",
    "Ann Nagar": "Anna Nagar", "Adyr": "Adyar", "Velchery": "Velachery",
    "KKNagar": "KK Nagar", "TNagar": "T Nagar", "Chrompt": "Chrompet",
    "Chrmpet": "Chrompet", "Chormpet": "Chrompet",
}
raw_groups = chn["AREA"].astype(str).str.strip().nunique()
chn["locality"] = chn["AREA"].astype(str).str.strip().replace(AREA_FIXES)
fixed_groups = chn["locality"].nunique()

print(f"distinct AREA values before normalising : {raw_groups}")
print(f"                                  after : {fixed_groups}")
print(f"=> {raw_groups - fixed_groups} were misspellings of a locality that")
print("   already existed. Left alone, each becomes its own spatial group and")
print("   the same locality spans the train/test split.\n")

chn["price_per_sqft"] = chn["SALES_PRICE"] / chn["INT_SQFT"]
chn = chn.dropna(subset=["price_per_sqft", "INT_SQFT", "locality"])
lo, hi = chn["price_per_sqft"].quantile([0.01, 0.99])
chn = chn[(chn["price_per_sqft"] >= lo) & (chn["price_per_sqft"] <= hi)]

print(f"Chennai after cleaning: {len(chn):,} rows")
print(f"  localities (= spatial blocks): {chn['locality'].nunique()}")
print(f"  {dict(chn['locality'].value_counts())}\n")

Xc = chn[["INT_SQFT", "N_BEDROOM", "N_BATHROOM", "N_ROOM"]]
yc = chn["price_per_sqft"].to_numpy()
gc = chn["locality"].to_numpy()

pc = Pipeline([("imp", SimpleImputer(strategy="median")),
               ("sc", StandardScaler())])
for name, est in [("random_forest", RandomForestRegressor(
                       n_estimators=120, random_state=42, n_jobs=-1)),
                  ("gradient_boosting", GradientBoostingRegressor(random_state=42))]:
    pipe = Pipeline([("pre", pc), ("m", est)])
    rand = cross_val_score(pipe, Xc, yc, cv=KFold(5, shuffle=True,
                                                  random_state=42),
                           scoring="r2", n_jobs=-1).mean()
    spat = cross_val_score(pipe, Xc, yc, cv=GroupKFold(5), groups=gc,
                           scoring="r2", n_jobs=-1).mean()
    print(f"  {name:<20} random {rand:>8.4f}   spatial {spat:>8.4f}   "
          f"gap {rand-spat:+.4f}")

print("\nA negative spatial R² means the model does WORSE than predicting the")
print(f"mean. With {chn['locality'].nunique()} localities there is almost nothing to")
print("generalise from, and the platform refuses to quote the random-split")
print("figure as accuracy.")
print("\nNote how much larger the gap is here than in Bengaluru. Fewer, bigger")
print("blocks make the leakage worse, not better.")

## 9 · The results reported as failures

Two outputs in this project are trained, measured, and then labelled unusable.
Both are read straight from the shipped model-comparison table.

In [ ]:
mc = data["model_comparison"]
display(mc.sort_values(["city", "spatial_cv_r2"], ascending=[True, False]))

print("Chennai's linear_regression scores ABOVE its tree models on spatial CV —")
print("with 7 blocks the trees overfit the blocks they saw. That inversion is")
print("why the shipped Chennai model is the simpler one.\n")

worst = mc.loc[mc["spatial_cv_r2"].idxmin()]
print(f"worst spatial R² in the project: {worst['algorithm']} on "
      f"{worst['city']} = {worst['spatial_cv_r2']}")
print("It is kept in the table rather than dropped, because a comparison that")
print("hides its failures is not a comparison.")

## 10 · Ward-level service accessibility

Per-ward scores for both cities. These are **weighted formulas, not machine
learning** — no dataset carries an observed "accessibility" label to train on,
and calling a formula ML would be the easiest lie in the project.

In [ ]:
for city in ["bengaluru", "chennai"]:
    wa = data.get(f"ward_analytics_{city}")
    if wa is None or wa.empty:
        continue
    cols = [c for c in wa.columns if c.endswith("_score")][:4]
    if not cols:
        cols = wa.select_dtypes("number").columns[:4].tolist()
    print(f"── {city}: {len(wa)} wards")
    display(wa[["ward_no"] + cols].describe().round(1) if "ward_no" in wa
            else wa[cols].describe().round(1))

## 11 · What this notebook did not do

Being explicit, because the omissions are deliberate:

* **No government record was fetched.** Khata, property tax, building permission
  and occupancy certificates are OTP-gated per-property portals. No public API
  exists for any of them, for anyone.
* **No ward population is published**, for the reason shown in section 5.
* **No FAR or floor limit is computed.** The governing instruments are cited in
  the platform but their clauses are not transcribed, so publishing a number
  would be a guess wearing a citation.
* **No flood risk score.** 391 reported locations with no return period, depth
  or drainage cannot support one.

The platform's value is calibrated honesty: it states what is verified, what is
indicative, and what nobody can obtain — and the last category is large.

---

Full platform, tests and data-source audit:
[github.com/ujwal-m-2006/AI-PROPERTY-URBAN-INTELLIGENCE](https://github.com/ujwal-m-2006/AI-PROPERTY-URBAN-INTELLIGENCE)